# NHL Total Goals Prediction

This notebook demonstrates the full ML pipeline for predicting total goals in NHL games:

1. **Data Collection** - Download NHL game data with caching
2. **Feature Engineering** - 20 features including rolling stats, rest days, streaks, **goalie stats**
3. **Model Training** - Optimized XGBoost that **beats the baseline**
4. **Cross-Validation** - Time-series CV for robust evaluation
5. **Analysis** - Feature importance and prediction quality

## Key Findings

After extensive experimentation, the **optimal configuration** is:
- Use **4 recent seasons** (2021-2025) - less regime change
- Use **window=20** for rolling features  
- Use **heavily regularized XGBoost** (max_depth=2, high reg_alpha/reg_lambda)
- Include **goalie save % and GAA** features

This achieves **+0.5-0.6% improvement over the baseline** (predicting the mean).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb

from src.data import build_dataset
from src.features import add_features
from src.model import (
    prepare_data,
    TrainingResult,
    plot_feature_importance,
    plot_predictions,
    save_model,
)

# Optimized XGBoost parameters (from hyperparameter tuning)
BEST_PARAMS = {
    'subsample': 0.7, 
    'reg_lambda': 2.0, 
    'reg_alpha': 1.0, 
    'n_estimators': 150, 
    'min_child_weight': 7, 
    'max_depth': 2, 
    'learning_rate': 0.01, 
    'colsample_bytree': 0.7, 
    'random_state': 42, 
    'n_jobs': -1, 
    'verbosity': 0
}

## 1. Download Data

We use **4 recent seasons (2021-2025)** for optimal performance. Using older data introduces regime change (rule changes, playing style evolution) that hurts prediction accuracy.

In [ ]:
# Use 4 recent seasons for best performance
seasons = ['20212022', '20222023', '20232024', '20242025']

# Build dataset (uses cache if available)
df_raw = build_dataset(seasons, use_cache=True)
print(f"\nTotal games: {len(df_raw):,}")
print(f"Date range: {df_raw['date'].min()} to {df_raw['date'].max()}")

## 2. Feature Engineering

Generate 20 features for each game:
- Rolling GF/GA averages (window=20 games)
- Rest days and back-to-back indicators
- Win streaks and percentages
- **Goalie rolling save % and GAA**

In [ ]:
# Add features with optimized 20-game rolling window and goalie stats
df = add_features(df_raw, window=20, min_games=3, include_goalies=True)

# Show feature columns
from src.model import get_feature_columns
feature_cols = get_feature_columns(df)
print(f"Generated {len(feature_cols)} features:")
for col in sorted(feature_cols):
    print(f"  - {col}")

In [ ]:
# Quick look at the data
df[['date', 'homeTeam', 'awayTeam', 'totalGoals', 'home_avg_GF', 'away_avg_GF', 'home_rest_days']].tail(10)

## 3. Train Optimized Model

Train XGBoost with heavily regularized parameters that beat the baseline.

In [ ]:
# Prepare data
X_train, X_test, y_train, y_test, feature_names = prepare_data(df, test_size=0.2)
print(f"Training: {len(X_train)} games, Testing: {len(X_test)} games")

# Calculate baseline
baseline_pred = np.full(len(y_test), y_train.mean())
baseline_mae = mean_absolute_error(y_test, baseline_pred)
print(f"\nBaseline MAE (predict mean): {baseline_mae:.4f}")

# Train optimized XGBoost
model = xgb.XGBRegressor(**BEST_PARAMS)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
improvement = (1 - mae / baseline_mae) * 100
print(f"XGBoost MAE: {mae:.4f}")
print(f"Improvement over baseline: {improvement:+.2f}%")

# Create result object for plotting
from sklearn.metrics import root_mean_squared_error
result = TrainingResult(
    model=model, model_type='XGBoost (Optimized)', feature_names=feature_names,
    y_test=y_test, y_pred=y_pred, mae=mae, 
    rmse=root_mean_squared_error(y_test, y_pred), 
    baseline_mae=baseline_mae
)

## 4. Cross-Validation

Use time-series cross-validation for more robust evaluation.

In [ ]:
# 5-fold time-series CV
tscv = TimeSeriesSplit(n_splits=5)
X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])

cv_results = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_full), 1):
    X_tr, X_te = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_tr, y_te = y_full.iloc[train_idx], y_full.iloc[test_idx]
    
    m = xgb.XGBRegressor(**BEST_PARAMS)
    m.fit(X_tr, y_tr)
    pred = m.predict(X_te)
    
    fold_mae = mean_absolute_error(y_te, pred)
    fold_baseline = mean_absolute_error(y_te, np.full(len(y_te), y_tr.mean()))
    imp = (1 - fold_mae / fold_baseline) * 100
    cv_results.append({'fold': fold, 'mae': fold_mae, 'baseline': fold_baseline, 'improvement': imp})
    print(f"Fold {fold}: MAE={fold_mae:.4f} vs Baseline={fold_baseline:.4f} ({imp:+.2f}%)")

cv_df = pd.DataFrame(cv_results)
print(f"\nCV Mean MAE: {cv_df['mae'].mean():.4f} ± {cv_df['mae'].std():.4f}")
print(f"CV Mean Improvement: {cv_df['improvement'].mean():+.2f}%")

## 5. Feature Importance

Which features are most predictive of total goals?

In [ ]:
# Plot feature importance - note goalie features in top 10!
plot_feature_importance(result, top_n=15)

## 6. Prediction Quality

In [ ]:
plot_predictions(result)

## 7. Save Model

In [ ]:
# Save the best model
save_model(result, Path("models/xgboost_v1.joblib"))

## Summary

| Metric | Value |
|--------|-------|
| Training Data | ~4,500 games (4 recent seasons) |
| Features | 20 (including goalie stats) |
| Best Model | XGBoost (heavily regularized) |
| Test MAE | 1.882 |
| Baseline MAE | 1.894 |
| **Improvement** | **+0.5-0.6%** |

### Key Insights

1. **Goalie features are important** - `home_goalie_gaa` is in the top 4 features
2. **Recent data works best** - Using only 2021-2025 reduces regime change
3. **Heavy regularization is critical** - max_depth=2 prevents overfitting
4. **Baseline is hard to beat** - NHL scoring is inherently noisy

### Optimized Parameters

```python
{
    'max_depth': 2,          # Very shallow trees
    'learning_rate': 0.01,   # Slow learning
    'n_estimators': 150,     # Moderate boosting rounds
    'reg_alpha': 1.0,        # L1 regularization
    'reg_lambda': 2.0,       # L2 regularization
    'subsample': 0.7,        # Row sampling
    'colsample_bytree': 0.7, # Feature sampling
    'min_child_weight': 7,   # Min samples per leaf
}
```

### Files Created
- `data/raw/*.csv` - Cached game data per season
- `data/goalies/goalie_stats.csv` - Starting goalie data for 14,650 games
- `models/xgboost_v1.joblib` - Trained model